This notebook is the blog / Chunk of my thoughts throughout the building process.

First I need to setup the pyproject.toml file, this file is a config file of what the project is and what resources we need.

Then I will make the basic structure of the whole project:

**notebooks folder:** annotate what I am doing throughout the project

**scripts folder:** data that I need for ML and sound analyze

**src/my_kws folder:**\
    `__init__.py`: The initial file\
    `augment.py`: data amplification\
    `dataset.py`: PyTorch Dataset\
    `features.py`: Extract characteristic\
    `io.py`: Audio I/O


Now here we have the code to only read audio file and output (mono information / sample rate / dtype)

In [ ]:
import soundfile as sf

def load_audio(
    path,
    target_sr,
    mono,
    target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")
    return data

if __name__ == "__main__":
    x = load_audio(r"D:\Downloads\oboe_samples\oboe_samples\Ob-ord-A4-mf.wav", 16000, True, 1.0)

shape=(297437,), sr=44100, dtype=float64


**Explain:**\
We define the function load_audio.\
`path`: path of the audio file\
`target_sr`: the target sample rate\
`mono`: if this is mono audio file\
`target_length_s`: the target length of file loaded

**After we run the code, the result:**\
`shape=(297437,)`: this is the mono file, one dimentional.\
`sr=44100`: the sample rate is 44100Hz, we need to resample to 16000Hz.\
`dtype=float64`: we need to convert this to float32.\

In [ ]:
import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF

def load_audio(
    path,
    target_sr,
    mono,
    target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")

    #************************
    #convert to float32
    data = data.astype(np.float32)

    #resample (if sr != target_sr)
    if sr != target_sr:
        waveform = torch.from_numpy(data).unsqueeze(0)
        waveform = AF.resample(waveform, sr, target_sr)
        data = waveform.squeeze(0).numpy()
    #************************
    print(f"after resample: shape={data.shape}, sr={target_sr}, dtype={data.dtype}")
    return data

if __name__ == "__main__":
    x = load_audio(r"D:\learnMLDSP\edgeaudio-ml\data\raw\esc50\audio\1-100032-A-0.wav", 16000, True, 1.0)

shape=(220500,), sr=44100, dtype=float64
after resample: shape=(80000,), sr=16000, dtype=float32


**Now we have two more steps: convert dtype and resample.**

soundfile is a library that i/o audio files (WAV/FLAC/OGG), it is based on libsndfile.<br>
`sf.read(path)` return two things:<br>
    data: numpy array, which is the PCM(Pulse Code Modulation) of the sample value.<br>
    Unify to [-1.0, 1.0] float64<br>
    sr: sample rate (Usually 44.1kHz)<br>
    data shape: mono for 1D (N,) ; stereo for 2D (N,2)




numpy is a python library for numerical computing, all audio sample lists/matrix are ndarray.<br>

np.float32 is a dtype (single-precision floating-point)

**Why converting to float32?**<br>
    Pytorch/torchaudio need float32 by default.
    float64 takes double memory and is slower on GPU


`import torch`<br>
`torch.from_numpy(data)` -- converts numpy array into a Pytorch tensor with 0-copy (shared memory)<br>
`.unsqueeze(0)` -- adds a length-1D at position 0D (80000,) -> (1, 80000)<br>
`.squeeze(0)` -- remove the length-1D at position 0<br>
`.numpy` -- converts tensor back into a numpy array (shared memory)<br>


**Why unsqueeze?**<br>
torchaudio API take waveform with (channels, samples), so you have to have a channel dimension, so for mono you have to belike: (N,) -> (1, N)

`import torchaudio.functional as AF`<br>
torchaudio is the Official-audio library for PyTorch.<br>

`.functional` (F or AF) is the functional interface (stateless functions).<br>
`.transforms` is the counterpart, which provides stateful `nn.Module` classes.<br>
**Functional for one-offs because you pass everything in as arguments every call; Transforms for repeated use in a pipeline or model.**
`AF.resample(waveform, orig_sr, new_sr` is to resample. resample from `orig_sr` to `new_sr`.<br>

Normal audio file 44100Hz, KWS use 16000Hz (16kHz for Nyquist, save 2.75x data)

Use polyphase filter + lowpass to avoid aliasing.

**Polyphase filter**: mathematically rearranges the computation so you only ever compute the samples you actually keep. (Much faster)

In [ ]:
def load_audio(
        path,
        target_sr,
        mono,
        target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")

    #convert to float32
    data = data.astype(np.float32)
    #*******************
    #check if mono
    if mono and data.ndim == 2:
        data = data.mean(axis=1)
    #*******************
    #resample (if sr != target_sr)
    if sr != target_sr:
        waveform = torch.from_numpy(data).unsqueeze(0) #shape: (1, n_samples)
        waveform = AF.resample(waveform, sr, target_sr)
        data = waveform.squeeze(0).numpy() #back to (n_samples,)

    print(f"after resample: shape={data.shape}, sr={target_sr}")
    
    return data

We make the audio mono if the audio is stereo.<br>
`if mono` -- means the user need mono as output, False if not.<br>
`data.ndim == 2` -- check if the data is 2D.<br>
`data.mean(axis=1)` -- find the mean of channel dimension.<br>

**axis=0**: reduces the first dimension / average along the "samples" dimension (Collapes N time steps into 1)<br>
**axis=1**: reduces the second dimension / average along the "channels" dimension (merges L and R channels into 1)<br>

In [ ]:
def load_audio(
        path,
        target_sr,
        mono,
        target_length_s
):
    data, sr = sf.read(path)
    print(f"shape={data.shape}, sr={sr}, dtype={data.dtype}")

    #convert to float32
    data = data.astype(np.float32)

    #check if mono
    if mono and data.ndim == 2:
        data = data.mean(axis=1)

    #resample (if sr != target_sr)
    if sr != target_sr:
        waveform = torch.from_numpy(data).unsqueeze(0) #shape: (1, n_samples)
        waveform = AF.resample(waveform, sr, target_sr)
        data = waveform.squeeze(0).numpy() #back to (n_samples,)
    print(f"after resample: shape={data.shape}, sr={target_sr}")
    #****************************
    #pad / crop
    target_n = int(target_sr * target_length_s)
    if len(data) > target_n:
        data = data[:target_n]
    elif len(data) < target_n:
        np.pad(data, (0, target_n - len(data)), mode='constant')
    #****************************
    return data

**Target Sample**: `target_n = int(target_sr * target_length_s)`<br>
use int() to be have a integer length.<br>

**Take samples from start to target_n**: `if len(data) > target_n: data = data[:target_n]`.<br>

**Pad**: `np.pad(array, pad_width, mode)`. If audio is too short then add 0 after the audio.<br>
Constant means you use constant number to fill the rest, 0 by default.


In [ ]:
#Root mean square
def rms(x : np.ndarray) -> float:
    float(np.sqrt(np.mean(x**2)))

**Root mean square**: the square root of the set's mean square.

In [ ]:
#Signal Noise Ratio SNR
def snr_db(signal: np.ndarray, noise: np.ndarray) -> float:
    #formula: 20 * log10(rms_signal / rms_noise)
    epsilon = 1e-10
    return 20 * np.log10(rms(signal) / (rms(noise) + epsilon))

**Signal Noise Ratio**: the ratio of signal power to noise power, often expressed in decibels. A ratio higher than 1:1 (greater than 0dB) indicates more signal than noise.

Use `(rms(signal) / (rms(noise) + epsilon))` instead of `(rms(signal) / rms(noise + epsilon))` because in Python `/` is prior to `+` when calculating.

Add epsilon(1e-10) after noise because if not then we will have a 1/0 calculation and it will return error or INF value, we use 1e-10 to limit it to about 200dB, which represent infinite.

# Framing

In [2]:
import numpy as np
def frame(waveform: np.ndarray, frame_length: int, hop_length: int) -> np.ndarray:
    T = len(waveform)
    n_frames = 1 + (T - frame_length) // hop_length

    frames = np.zeros((n_frames, frame_length), dtype = waveform.dtype)
    for i in range(n_frames):
        start = i*hop_length
        end = i*hop_length + frame_length
        frames[i] = waveform[start : end]
    return frames

**Param Types**:<br>
waveform: shape (T,)<br>
frame_length: samples per frame<br>
hop_length: samples between frame starts<br>

`T = len(waveform)` save the 1D array, equal to `waveform.shape[0]`<br>
Why T? For readability, not optimization.

`n_frames = 1 + (T - frame_length) // hop_length`<br>
`//` means divede(int), like 3/2=1; while `/` means divede(float)<br>
    Frame0: start at [0, frame_length]<br>
    Frame1: start at i * hop_length<br>
    Framei: start at i * hop_length + frame_length <=T<br>
    Max: (T - frame_length) // hop_length<br>

Edge Case: T = 1000, frame_length = 400, hop_length = 160<br>
n_frames = 1 + (1000-400)//160 = 4<br>
frame3: [480, 880)<br>
abandoned: [880, 1000)<br>


`frames = np.zeros((n_frames, frame_length), dtype = waveform.dtype)`

use `np.zeros((n_frames, frame_length)` to create a All-0 matrix, make dtype the same as waveform (float32).

Fill frames: `for i in range(n_frames):`<br>
start and end means the start and end of frame i<br>
`waveform[start:end]` returns numpy (frame_length,) 1D array<br>
then write the 1D array to `frames[i]`.

# Window function (Hann window)

In [ ]:
#Window function Hanning window
def window(frames: np.ndarray, window_type: str = "hann") -> np.ndarray:
    N = frames.shape[1]
    if window_type == "hann":
        n = np.arange(N)
        w = 0.5 * (1 - np.cos(2 * np.pi * n / (N - 1)))
    else:
        raise ValueError("Not Supported!")
    
    return frames * w

`frames.shape[1]` means N equal to the 1D array of frames<br>
`n = np.arange(N)` generate a array from 0 to N-1<br>

**Hann Formula**: `0.5 * (1 - np.cos(2 * np.pi * n / (N - 1)))`<br>
    This return float64 so we need to manually convert to float32

`w = w.astype(frames.dtype)` convert from float64 to float32

# STFT: Short time Fourier Transform

**Params**:<br>
`win_length=160`: how many samples per frame (25ms @ 16kHz), determine time resolution.<br>
`n_fft=512`: determines frequency resolution.(bin length = 16000/512 = 31Hz)<br>
`hop_length=160`: shift of frame(10ms), determines the density of frames on timeline.

In [ ]:
def stft(waveform: np.ndarray, n_fft: int = 512, hop_length: int = 160, win_length: int = 400) -> np.ndarray:
    frames = frame(waveform, win_length, hop_length)

    frames = window(frames)

    padded = np.zeros((frames.shape[0], n_fft), dtype=frames.dtype)
    padded[:, :win_length] = frames

    spectrum = np.fft.rfft(padded, axis=1).astype(np.complex64)

    return spectrum

`frames = frame(waveform, win_length, hop_length)`<br>
Calculation:<br>
frame formula:<br>
    n_frames = 1 + (T - win_length) // hop_length<br>
             = 1 + (16000 - 400) // 160<br>
             = 1 + 15600 // 160<br>
             = 1 + 97<br>
             = 98<br>

overlap:<br>
    overlap = (win_length - hop_length) / win_length<br>
            = (400 - 160) / 400<br>
            = 60%<br>

`frames = window(frames)`<br>
Prevent spectral leakage, suppress the wave edge to 0 with Hann window.

`padded = np.zero((n_frames, n_fft), dtype=frame.dtype)`<br>
pad/fill 0 after short audio frames (400->512), make FFT more accurate on frequency.<br>

`spectrum = np.fft.rfft(padded, axis=1).astype(np.complex64)`<br>
Do rfft for every frame, rfft only return `n_fft//2 + 1 = 257` bins, which is first half of FFT, save calculation because the FFT second half is abundant.

**Final Shape (Spectrogram)**:<br>
(98, 257)<br>
column: 98, timeline, frame per column (10ms)
row: 257, frequency, 0Hz -> 8000Hz, 31Hz for every bin.